# Now to actually make map

In [29]:
import os
import json

import numpy as np
import pandas as pd

from ipywidgets import Dropdown

from bqplot import Lines, Figure, LinearScale, DateScale, Axis

from ipyleaflet import Map, GeoJSON, WidgetControl

import plotly.graph_objects as go

In [30]:
%run graph_programs/female_percentage.py


In [31]:
# data = pd.read_json(os.path.abspath("nations.json"))

# converting shpae file to geojson
import geopandas as gpd

geo_json_path = os.path.join("geos", "cc20_us_03102025_500k_small.geojson")
gdf_small = gpd.read_file(geo_json_path)  # Load the existing GeoJSON file
    

In [32]:
print(gdf_small.columns)
geoids = gdf_small["GEOID"].astype(str).tolist()
print(geoids) 

bounds = gdf_small.total_bounds

Index(['CCN20', 'GEOID', 'STCD', 'STFIPS', 'STAB', 'D001TPOP', 'D0020004',
       'D0030509', 'D0041014', 'D0051519',
       ...
       'D154VSEAS', 'D155VOTHR', 'D156HVACR', 'D157RVACR', 'D158OCCHU',
       'D159OCCHUO', 'D160OCCHUR', 'DIVERSITY', 'DEPRATIO', 'geometry'],
      dtype='str', length=168)
['0101001', '0101002', '0101003', '0101004', '0101005', '0101006', '0101007', '0101008', '0101009', '0101010', '0101011', '0101012', '0101013', '0101014', '0101015', '0101016', '0102001', '0102002', '0102003', '0102004', '0102005', '0102006', '0102007', '0102008', '0102009', '0102010', '0102011', '0102012', '0102013', '0102014', '0102015', '0102016', '0102017', '0102018', '0103001', '0103002', '0103003', '0103004', '0103005', '0103006', '0103007', '0103008', '0103009', '0103010', '0103011', '0103012', '0103013', '0103014', '0103015', '0103016', '0103017', '0104001', '0104002', '0104003', '0104004', '0104005', '0104006', '0104007', '0104008', '0104009', '0104010', '0104011', '0104012', '

In [33]:
demographic_var = ["D001TPOP", "D025MALE", "D049FEMALE"]  # Replace with the actual column name for the demographic variable

In [34]:
# cleaning data for proof
data = gdf_small[['GEOID'] + demographic_var][gdf_small["GEOID"].astype(str).isin(geoids)]
print(data.head())

     GEOID  D001TPOP  D025MALE  D049FEMALE
0  0101001     55599     27908       27691
1  0101002     67916     32845       35071
2  0101003     34347     16594       17753
3  0101004     45525     22560       22965
4  0101005     36757     18254       18503


In [35]:
m = Map(zoom=5)

# this makes the map tiles
geo = GeoJSON(
    data=gdf_small.__geo_interface__,
    style={"fillColor": "white", "weight": 0.5},
    hover_style={"fillColor": "#1f77b4"},
    name="GEOID",
)

m.add(geo)

# Fit map to bounds: [[south, west], [north, east]]
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

m

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

In [36]:
# Highlight overlay — starts empty, only ever holds the small selected subset
highlight_layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"fillColor": "#D85A30", "color": "black", "weight": 2, "fillOpacity": 0.7},
)


m.add_layer(highlight_layer)

In [37]:
from bqplot import Figure, Label, LinearScale

# Create a dummy scale (required for the Figure)
scale = LinearScale()

# Create a label mark with your single number
label = Label(
    text=['42'],
    x=[0.5], 
    y=[0.5], 
    scales={'x': scale, 'y': scale}, 
    font_size='36px', )

label_side = Label(
    text=['42'],
    x=[0.5], 
    y=[0.5], 
    scales={'x': scale, 'y': scale}, 
    font_size='36px', )

# Display the figure
fig = Figure(
    marks=[label],
    layout={"width": "100px", "height": "80px"},  # controls the actual rendered size
    fig_margin={"top": 5, "bottom": 5, "left": 5, "right": 5},  # shrink default whitespace padding
    min_aspect_ratio=0, max_aspect_ratio=100,  # prevents bqplot forcing a min size to preserve aspect ratio
)


fig

Figure(fig_margin={'top': 5, 'bottom': 5, 'left': 5, 'right': 5}, layout=Layout(height='80px', width='100px'),…

In [38]:
fig_percent = make_female_percentage_graph()
fig_percent.show()

In [39]:
def update_figure (geoid, var_of_interest):
    # Filter the data for the selected GEOID
    filtered_data = data[data['GEOID'] == geoid]
    
    # Get the value of the demographic variable
    value = filtered_data[var_of_interest].values[0] if not filtered_data.empty else 'N/A'
    
    # Update the label text
    label.text = [str(value)]


In [40]:
update_figure('0101002', 'D001TPOP')  # Example GEOID, replace with actual GEOID as needed
fig

Figure(fig_margin={'top': 5, 'bottom': 5, 'left': 5, 'right': 5}, layout=Layout(height='80px', width='100px'),…

In [41]:
# adding widget on hover
widget_control1 = WidgetControl(widget=fig, position="bottomright")

m.add(widget_control1)

def on_hover(event, feature, **kwargs):
    global geoid
    geoid = feature["properties"]["GEOID"]
    update_figure(geoid, 'D001TPOP')  # Update the figure with the selected GEOID and demographic variable

geo.on_hover(on_hover)

m

Map(center=[np.float64(32.615677443261795), np.float64(-86.68073621090903)], controls=(ZoomControl(options=['p…

Just to cmodify on the side

In [42]:
# making. a line graph for demo variable

total_pop_fig = Figure(
    marks=[Lines(x=data['GEOID'], y=data[demographic_var], scales={'x': LinearScale(), 'y': LinearScale()})],
    axes=[Axis(label='GEOID', scale=LinearScale()), Axis(label='Total Population', scale=LinearScale(), orientation='vertical')],
    title='Total Population by GEOID'
)

def update_total_pop_fig(geoids):
    # geoids can be a single value or a list — normalize to a list
    if not isinstance(geoids, (list, set, tuple)):
        geoids = [geoids]

    filtered_data = data[data['GEOID'].isin(geoids)]

    if filtered_data.empty:
        total_pop_fig.marks[0].x = []
        total_pop_fig.marks[0].y = []
    else:
        total_pop_fig.marks[0].x = filtered_data['GEOID'].tolist()
        total_pop_fig.marks[0].y = filtered_data[demographic_var].tolist()

In [43]:
from ipywidgets import Button, Layout
import ipyleaflet
# I need a reset selectin button 
def reset_selection(button):
    selected_geoids.clear()  # Clear the selection
    update_fig_side([])  # Update the side figure with no selection
    update_highlight_layer([])  # Clear the highlight layer
    reset_female_percentage_graph(fig_percent)  # Reset
    update_right_panel(fig_percent)  # Update the right panel with the reset graph
    
reset_btn = Button(
    description='Reset Selection', 
    button_style='info',
    layout=Layout(width='auto', height='auto')
)

reset_btn.on_click(reset_selection)

widget_control = ipyleaflet.WidgetControl(widget=reset_btn, position='topright')
m.add_control(widget_control)

m

Map(center=[np.float64(32.615677443261795), np.float64(-86.68073621090903)], controls=(ZoomControl(options=['p…

In [44]:
from ipywidgets import HBox
import plotly.graph_objects as go

# Convert the static Figure into a live widget so it can sit in an HBox
right_panel = go.FigureWidget(fig_percent)
right_panel.layout.width = 350
right_panel.layout.height = 225

m_layout = HBox([m, right_panel])
m_layout

In [45]:
def update_right_panel_2(fig_percent):
    global m_layout
    
    # Convert the static Figure into a live widget so it can sit in an HBox
    right_panel = go.FigureWidget(fig_percent)
    right_panel.layout.width = 350
    right_panel.layout.height = 225

    m_layout = HBox([m, right_panel])
    
def update_right_panel(fig_percent):
    global right_panel
    right_panel.data = []
    for trace in fig_percent.data:
        right_panel.add_trace(trace)
    right_panel.layout.update(fig_percent.layout.to_plotly_json())

on click events
https://medium.com/swlh/how-to-use-mouse-events-on-ipyleaflet-4d002097efc0

In [46]:
from ipywidgets import Output
from IPython.display import display

debug_out = Output()
display(debug_out)

def handle_click(**kwargs):
    with debug_out:
        print(kwargs)

geo.on_click(handle_click)
display(m_layout)

Output()

In [47]:
def update_fig_side(geoids):
    # geoids can be a single value or a list — normalize to a list
    if not isinstance(geoids, (list, set, tuple)):
        geoids = [geoids]

    filtered_data = data[data['GEOID'].isin(geoids)]

    if filtered_data.empty:
        value = 'N/A'
    else:
        value = filtered_data[demographic_var].values  # array of values, one per selected region
        # decide here how to combine them if multiple selected, e.g.:
        # value = filtered_data[demographic_var].mean()  # average across selection
        # or keep as a list to show all selected values in the label/chart

    # Update the label text
    label_side.text = [str(value)]

In [48]:
def update_highlight_layer(geoids):
    # geoids can be a single value or a list — normalize to a list
    if not isinstance(geoids, (list, set, tuple)):
        geoids = [geoids]

    # Filter the GeoDataFrame for the selected GEOIDs
    filtered_gdf = gdf_small[gdf_small['GEOID'].astype(str).isin(geoids)]

    # Update the highlight layer with the new GeoJSON data
    highlight_layer.data = filtered_gdf.__geo_interface__

In [49]:
print(data.head())

     GEOID  D001TPOP  D025MALE  D049FEMALE
0  0101001     55599     27908       27691
1  0101002     67916     32845       35071
2  0101003     34347     16594       17753
3  0101004     45525     22560       22965
4  0101005     36757     18254       18503


In [50]:
# now doing percentage of male and female population
data["D025MALE_PERCENT"] = (data["D025MALE"] / data["D001TPOP"]) * 100
data["D049FEMALE_PERCENT"] = (data["D049FEMALE"] / data["D001TPOP"]) * 100
print(data[["GEOID", "D025MALE_PERCENT", "D049FEMALE_PERCENT"]].head())

     GEOID  D025MALE_PERCENT  D049FEMALE_PERCENT
0  0101001         50.195147           49.804853
1  0101002         48.361211           51.638789
2  0101003         48.312808           51.687192
3  0101004         49.555189           50.444811
4  0101005         49.661289           50.338711


In [51]:
from graph_programs.female_percentage import update_female_percentage_graph

fig_percent_1 = update_female_percentage_graph(data, gdf_small, fig_percent)
fig_percent_1.show()

Now I want to incporate this into the graph

In [52]:
# initilaizing 
from graph_programs.female_percentage import update_female_percentage_graph

selected_geoids = set()

def handle_click(event=None, feature=None, **kwargs):
    global fig_percent
    geoid = feature["properties"]["GEOID"]

    if geoid in selected_geoids:
        selected_geoids.discard(geoid)
    else:
        selected_geoids.add(geoid)

    filtered = gdf_small[gdf_small['GEOID'].astype(str).isin(selected_geoids)]
    fig_percent = update_female_percentage_graph(
        demographics_data = data, 
        geoids_df = filtered, 
        fig_percent = fig_percent)
    update_right_panel(fig_percent)
    update_highlight_layer(list(selected_geoids))

geo.on_click(handle_click)

display(m_layout)


In [54]:
fig_percent.show()